In [ ]:
# parameters
# BINDER_FAST: set N=6, n_steps=30, n_scan=15 for fast cloud execution
N = 10            # Hilbert space truncation (Fock dimension)
omega0_GHz = 5.0  # bare cavity frequency (GHz)
K_GHz = 0.2       # Kerr nonlinearity (GHz)
T_ns = 100        # gate time in nanoseconds
n_steps = 80      # time slices for propagator construction
n_scan = 30       # number of drive amplitudes in the parameter scan
TWO_PI = 2 * 3.141592653589793

In [ ]:
# hide
import numpy as np
import qutip as qt
import matplotlib.pyplot as plt
%matplotlib widget

from bosonic_gates.gates import snap_operator, snap_unitary_ideal
from bosonic_gates.driven_kerr import DrivenKerrConfig, make_H0, make_operators
from bosonic_gates.driven_kerr.core import make_H_drive_td

TWO_PI = 2 * np.pi

In [ ]:
# hide
def build_propagator(H, tlist, N):
    """Compute the N×N unitary propagator by evolving each basis ket.

    Parameters
    ----------
    H : qt.Qobj or list
        Hamiltonian — either a static qt.Qobj or a QuTiP time-dependent list
        [H0, [H_ctrl, coeff_fn]].
    tlist : np.ndarray
        Times at which to evaluate the evolution.  The propagator is taken
        at the final time tlist[-1].
    N : int
        Hilbert space dimension.

    Returns
    -------
    U : qt.Qobj  (dims [[N],[N]])
    """
    cols = []
    opts = qt.Options(nsteps=10000)
    for j in range(N):
        psi0 = qt.basis(N, j)
        result = qt.sesolve(H, psi0, tlist, options=opts)
        cols.append(result.states[-1].full().flatten())
    U_mat = np.column_stack(cols)
    U = qt.Qobj(U_mat)
    U.dims = [[N], [N]]
    return U

## Module 5a: Propagators and Gate Fidelity

**Learning objectives:**
- Construct the unitary time-evolution propagator $\hat{U}(T)$ numerically from QuTiP `sesolve`
- Verify unitarity and interpret the propagator as a matrix on Fock space
- Compute gate fidelity using the Nielsen trace formula
- Manually scan a single pulse parameter to explore the fidelity landscape
- Understand the piecewise-constant pulse model used by GRAPE

---

**Sections:**
[1 Time-Evolution Propagator](#sec1) ·
[2 Gate Fidelity](#sec2) ·
[3 Parameter Scan](#sec3) ·
[4 Piecewise-Constant Pulses](#sec4) ·
[Exercises](#exercises)

<a id="sec1"></a>
## 1  The Time-Evolution Propagator

The state of a quantum system at time $T$ is related to its initial state by the **unitary propagator** $\hat{U}(T)$:

$$|\psi(T)\rangle = \hat{U}(T)|\psi(0)\rangle.$$

For a **time-independent** Hamiltonian the propagator has the closed form

$$\hat{U}(T) = e^{-i\hat{H}T/\hbar}.$$

For a **time-dependent** $\hat{H}(t)$ there is no closed form in general.  The formal
solution is given by the time-ordered Dyson series:

$$\hat{U}(T) = \mathcal{T}\exp\!\left(-\frac{i}{\hbar}\int_0^T \hat{H}(t)\,dt\right)
= 1 + \sum_{n=1}^\infty \frac{(-i/\hbar)^n}{n!}\int_0^T\!\cdots\!\int_0^T
\mathcal{T}\bigl[\hat{H}(t_1)\cdots\hat{H}(t_n)\bigr]\,dt_1\cdots dt_n,$$

where $\mathcal{T}$ is the time-ordering operator.  In practice we integrate
numerically using QuTiP's `sesolve`.

### Column-by-column construction

The propagator matrix element $U_{ij} = \langle i|\hat{U}(T)|j\rangle$ is obtained
by propagating the $j$-th basis state $|j\rangle$ and reading off the $i$-th component:

$$\hat{U}(T) = \sum_{j=0}^{N-1} |\psi_j(T)\rangle\langle j|, \qquad
|\psi_j(T)\rangle = \hat{U}(T)|j\rangle.$$

This requires $N$ independent `sesolve` calls but generalises immediately to any $\hat{H}(t)$.

**Unitarity check:** $\hat{U}^\dagger \hat{U} = \mathbf{1}$ to machine precision.  Any
deviation indicates insufficient time resolution in `tlist` or a non-Hermitian bug in $\hat{H}$.

In [ ]:
# Build the driven-Kerr Hamiltonian (no drive: epsilon=0)
cfg = DrivenKerrConfig(N=N, omega0=TWO_PI * omega0_GHz * 1e9,
                       K=TWO_PI * K_GHz * 1e9)
H0 = make_H0(cfg)

# Propagate over T_ns nanoseconds
tlist = np.linspace(0, T_ns * 1e-9, n_steps + 1)

print(f"N = {N},  T = {T_ns} ns,  n_steps = {n_steps}")
print(f"omega0 / (2pi) = {cfg.omega0 / TWO_PI / 1e9:.3f} GHz")
print(f"K / (2pi) = {cfg.K / TWO_PI / 1e9:.3f} GHz")

# Build propagator column-by-column
U = build_propagator(H0, tlist, N)

# Verify unitarity: U†U should be the identity
UdagU = U.dag() * U
identity_err = (UdagU - qt.qeye(N)).norm()
print(f"\n||U†U - I|| = {identity_err:.2e}  (should be ~machine epsilon)")

In [ ]:
# Visualise the propagator as a phase heatmap
U_mat = U.full()
phase = np.angle(U_mat)
amp   = np.abs(U_mat)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

im0 = axes[0].imshow(amp, origin='lower', cmap='viridis', vmin=0, vmax=1)
plt.colorbar(im0, ax=axes[0])
axes[0].set_title(r"$|U_{ij}|$  (amplitude)")
axes[0].set_xlabel(r"$j$ (column = initial state)")
axes[0].set_ylabel(r"$i$ (row = final state)")

im1 = axes[1].imshow(phase, origin='lower', cmap='hsv', vmin=-np.pi, vmax=np.pi)
plt.colorbar(im1, ax=axes[1], label='phase (rad)')
axes[1].set_title(r"$\arg(U_{ij})$  (phase)")
axes[1].set_xlabel(r"$j$")
axes[1].set_ylabel(r"$i$")

plt.suptitle(
    rf"Propagator $\hat{{U}}(T={T_ns}\,\mathrm{{ns}})$ — Kerr oscillator, no drive",
    fontsize=12)
plt.tight_layout()
plt.show()

print("\nDiagonal elements (each Fock state acquires a phase):")
for n in range(min(N, 8)):
    print(f"  U[{n},{n}] = {U_mat[n,n]:.4f}   phase = {np.angle(U_mat[n,n]):.4f} rad")

**Observation.** For the undriven Kerr oscillator, the propagator is diagonal:
each Fock state $|n\rangle$ acquires only a phase $e^{-i\phi_n}$ where
$\phi_n = \omega_0 n T - \tfrac{K}{2}n(n-1)T$.  The Kerr term makes these
phases non-uniform — this is the key nonlinearity that optimal control exploits
to engineer non-trivial unitaries like SNAP gates.

---
*__Lab note.__ In experiment you cannot measure $\hat{U}$ directly.
Gate tomography reconstructs $\hat{U}$ from the density matrices of a
complete set of input states (quantum process tomography, QPT).  For an
$N$-level system, QPT requires $N^2$ input states and $N^2$ measurements — a
resource cost that scales as $N^4$, making it expensive for large Hilbert spaces.*

<a id="sec2"></a>
## 2  Gate Fidelity — the Nielsen Formula

How close is a realized gate $\hat{U}$ to its target $\hat{U}_{\rm target}$?
The **process fidelity** (Nielsen & Chuang, Eq. 9.173) answers this:

$$\boxed{F(\hat{U}, \hat{U}_{\rm target}) = \frac{|\mathrm{Tr}(\hat{U}^\dagger \hat{U}_{\rm target})|^2}{d^2}}$$

where $d$ is the Hilbert space dimension.

**Key properties:**

| Condition | $F$ |
|---|---|
| $\hat{U} = e^{i\phi}\hat{U}_{\rm target}$ (equal up to global phase) | $1$ |
| $\hat{U} = \hat{U}_{\rm target}$ (identical) | $1$ |
| Maximally distant unitaries | $0$ |
| Completely random unitary | $\approx 1/d$ |

The **infidelity** $1 - F$ is the natural optimization objective: minimizing
$1-F$ is equivalent to maximizing $F$.  For high-quality gates, $1-F \ll 1$
and the infidelity budget can be apportioned across error channels
(decoherence, pulse calibration, leakage).

**Efficient computation** uses the Hilbert–Schmidt inner product:

$$F = \frac{|\mathrm{Tr}(\hat{U}^\dagger \hat{U}_{\rm target})|^2}{d^2}
    = \frac{\bigl|\langle \hat{U}, \hat{U}_{\rm target}\rangle_{\rm HS}\bigr|^2}{d^2}.$$

No matrix inversion is needed; a single matrix multiply and trace suffices.

In [ ]:
def gate_fidelity(U_achieved: qt.Qobj, U_target: qt.Qobj) -> float:
    """Process fidelity F = |Tr(U†V)|² / d² (Nielsen formula)."""
    d = U_target.shape[0]
    overlap = (U_achieved.dag() * U_target).tr()
    return float(np.abs(overlap)**2 / d**2)


# Target: SNAP gate with phase pi on Fock level 2
thetas = np.zeros(N)
thetas[2] = np.pi
U_target = snap_operator(N, thetas)

# Identity gate: all phases zero
U_identity = qt.qeye(N)
U_identity.dims = [[N], [N]]

F_id_vs_target = gate_fidelity(U_identity, U_target)
F_target_vs_target = gate_fidelity(U_target, U_target)

print("=== Gate Fidelity Checks ===")
print(f"F(identity, SNAP[theta_2=pi]) = {F_id_vs_target:.6f}  (should be < 1)")
print(f"F(SNAP[theta_2=pi], SNAP[theta_2=pi]) = {F_target_vs_target:.6f}  (should be 1)")

print("\nAnalytic check:")
# Identity has trace N; target has trace N-2 (levels 0,1,3,...,N-1 contribute 1 each, level 2 contributes e^(i*pi)=-1)
tr_analytic = N - 2  # N-1 levels contribute +1, level 2 contributes -1
F_analytic = abs(tr_analytic)**2 / N**2
print(f"  Tr(I† SNAP) = {(U_identity.dag()*U_target).tr():.4f}  (analytic: {N-2})")
print(f"  F analytic = {F_analytic:.6f}")

In [ ]:
# How does F depend on a phase error on theta[2]?
delta_theta_vals = np.linspace(0, 2 * np.pi, 300)
F_vals = []
for dt in delta_theta_vals:
    th = np.zeros(N)
    th[2] = np.pi + dt
    F_vals.append(gate_fidelity(snap_operator(N, th), U_target))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(delta_theta_vals, F_vals, lw=2)
ax.axhline(1.0, color='gray', ls='--', lw=1)
ax.axhline(0.0, color='gray', ls=':', lw=1)
ax.set_xlabel(r"Phase error $\delta\theta_2$ (rad)")
ax.set_ylabel(r"Gate fidelity $F$")
ax.set_title(r"$F$ vs phase error on SNAP level $|2\rangle$")
ax.set_xlim(0, 2 * np.pi)
ax.set_xticks([0, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi],
              [r'$0$', r'$\pi/2$', r'$\pi$', r'$3\pi/2$', r'$2\pi$'])
plt.tight_layout()
plt.show()

print("Sample points:")
for dt in [0, 0.1, np.pi/4, np.pi/2, np.pi]:
    th = np.zeros(N); th[2] = np.pi + dt
    print(f"  delta_theta = {dt:.4f} rad  ->  F = {gate_fidelity(snap_operator(N, th), U_target):.6f}")

**Observation.** The fidelity follows $F = \cos^2(\delta\theta/2)$ for a single-level SNAP
error (analytic: the trace picks up one factor of $e^{i\,\delta\theta}$ from level 2,
all other levels contribute $+1$).  An error of $\delta\theta = 0.1\,\mathrm{rad}$ gives
$1-F \approx 2.5\times10^{-3}$ — a useful budget estimate for pulse calibration.

---
*__Lab note.__ This metric is used throughout optimal control: GRAPE, CRAB, and Krotov all
optimize $F$ via gradient ascent or iterative updates on the pulse parameters.  The next
two notebooks show how to do this systematically rather than scanning by hand.*

<a id="sec3"></a>
## 3  Manual Parameter Scan — the Fidelity Landscape

Before running a full optimizer it is instructive to scan a **single scalar control
parameter** and observe how the fidelity varies.  This builds intuition for the
landscape that GRAPE navigates in high dimensions.

Here we drive the Kerr cavity at its bare frequency $\omega_0$ with a constant amplitude
$\varepsilon$ for a fixed time $T$ and measure the fidelity against the target SNAP gate.
The time-dependent Hamiltonian in the lab frame is:

$$\hat{H}(t) = \omega_0\hat{n} - \frac{K}{2}\hat{n}(\hat{n}-1) + \varepsilon\cos(\omega_0 t)(\hat{a}+\hat{a}^\dagger)$$

where $\varepsilon$ is the drive amplitude in units of angular frequency.
Scanning $\varepsilon$ produces a Rabi-like oscillation in fidelity:
small $\varepsilon$ barely moves the state, large $\varepsilon$ over-rotates it,
and at special values the gate fidelity with a simple target can reach local maxima.

This scan is the **manual, 1-D precursor** to GRAPE: GRAPE generalises the search to
$n_{\rm steps}$ independent pulse amplitudes optimized simultaneously via gradient ascent.

In [ ]:
# Target: simple SNAP with phase pi on level 1 only (easier to hit with a flat pulse)
thetas_scan = np.zeros(N)
thetas_scan[1] = np.pi
U_scan_target = snap_operator(N, thetas_scan)

# Drive amplitudes to scan (rad/s, convert from GHz)
epsilon_GHz_vals = np.linspace(0.0, 0.5, n_scan)
epsilon_vals = epsilon_GHz_vals * TWO_PI * 1e9

tlist_scan = np.linspace(0, T_ns * 1e-9, n_steps + 1)

fidelities_scan = []
for eps in epsilon_vals:
    cfg_eps = DrivenKerrConfig(
        N=N,
        omega0=TWO_PI * omega0_GHz * 1e9,
        K=TWO_PI * K_GHz * 1e9,
        epsilon=eps,
    )
    H_td = make_H_drive_td(cfg_eps)
    U_eps = build_propagator(H_td, tlist_scan, N)
    F_eps = gate_fidelity(U_eps, U_scan_target)
    fidelities_scan.append(F_eps)
    print(f"  eps/(2pi) = {eps/TWO_PI/1e9:.3f} GHz  ->  F = {F_eps:.4f}")

In [ ]:
best_idx = int(np.argmax(fidelities_scan))
best_eps = epsilon_GHz_vals[best_idx]
best_F   = fidelities_scan[best_idx]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(epsilon_GHz_vals, fidelities_scan, 'o-', lw=2, ms=5)
ax.axvline(best_eps, color='crimson', ls='--', label=rf"best $\varepsilon = {best_eps:.3f}$ GHz, $F={best_F:.3f}$")
ax.set_xlabel(r"Drive amplitude $\varepsilon/2\pi$ (GHz)")
ax.set_ylabel(r"Gate fidelity $F$")
ax.set_title(
    rf"Fidelity scan: SNAP $\theta_1=\pi$, $T={T_ns}$ ns, $K/2\pi={K_GHz}$ GHz"
)
ax.set_ylim(-0.05, 1.05)
ax.legend()
plt.tight_layout()
plt.show()

print(f"\nBest amplitude: eps/(2pi) = {best_eps:.4f} GHz")
print(f"Best fidelity:  F = {best_F:.6f}")
print(f"Infidelity:  1-F = {1-best_F:.6e}")

**Observation.** The fidelity landscape shows Rabi-like oscillations:
the driven cavity undergoes approximate Rabi oscillations on the $|1\rangle \leftrightarrow |2\rangle$
transition dressed by the Kerr nonlinearity.  The landscape is non-convex and has multiple
local maxima — a single-parameter scan can already identify useful operating points.

---
*__Lab note.__ This 1-D scan illustrates the core challenge of pulse optimization:
a realistic pulse has $n_{\rm steps} \times n_{\rm ctrl}$ free parameters,
and the fidelity landscape is high-dimensional and non-convex.
GRAPE navigates this landscape using automatic differentiation to compute the
full gradient $\partial F/\partial u_k$ at every time slice simultaneously,
making it far more efficient than any manual scan.*

<a id="sec4"></a>
## 4  The Piecewise-Constant Pulse Model

Optimal control algorithms parametrize the control field as a
**piecewise-constant** (PWC) pulse: the time interval $[0, T]$ is divided into
$n_{\rm steps}$ slices of duration $\delta t = T/n_{\rm steps}$, and the
control amplitude is held constant within each slice:

$$u(t) = u_k \quad \text{for } t \in [(k-1)\delta t,\, k\delta t), \quad k = 1, \ldots, n_{\rm steps}.$$

The total Hamiltonian in slice $k$ is

$$\hat{H}(t_k) = \hat{H}_0 + u_k\,\hat{H}_{\rm ctrl},$$

and the propagator factorises as an **ordered product** of matrix exponentials:

$$\hat{U}(T) = \hat{U}_{n_{\rm steps}} \cdots \hat{U}_2\,\hat{U}_1, \qquad
\hat{U}_k = e^{-i\hat{H}(t_k)\delta t}.$$

**Why this works:** each $\hat{U}_k$ is exactly unitary; the product of
unitaries is unitary; and the PWC approximation converges to the true propagator
as $\delta t \to 0$ (first-order Lie–Trotter expansion).

GRAPE finds the $\{u_k\}$ that maximize $F$ by computing the gradient
$\partial F/\partial u_k$ through this product — either analytically
(backward co-state propagation) or via automatic differentiation (JAX).

In [ ]:
# Demonstrate piecewise-constant propagator construction
from scipy.linalg import expm

def build_propagator_pwc(H0_mat, H_ctrl_mat, pulse, T, n_steps):
    """Build propagator by product of matrix exponentials for a PWC pulse.

    Parameters
    ----------
    H0_mat, H_ctrl_mat : np.ndarray  (N x N complex)
    pulse : np.ndarray, shape (n_steps,)  — amplitude u_k for each slice
    T : float, n_steps : int

    Returns
    -------
    U : np.ndarray  (N x N complex)
    """
    N = H0_mat.shape[0]
    dt = T / n_steps
    U = np.eye(N, dtype=complex)
    for k in range(n_steps):
        H_k = H0_mat + pulse[k] * H_ctrl_mat
        U = expm(-1j * H_k * dt) @ U
    return U


# Simple Kerr oscillator, no drive (flat pulse = 0)
a_op, adag_op, _ = make_operators(N)
H0_mat  = H0.full()
Hc_mat  = (a_op + adag_op).full()

T_test = T_ns * 1e-9
flat_pulse = np.zeros(n_steps)  # zero drive

U_pwc = build_propagator_pwc(H0_mat, Hc_mat, flat_pulse, T_test, n_steps)

# Compare with sesolve result (should match well for n_steps = 80)
U_sesolve_mat = U.full()  # from Section 1

err = np.max(np.abs(U_pwc - U_sesolve_mat))
print(f"PWC propagator vs sesolve: max |difference| = {err:.4e}")
print(f"(n_steps={n_steps}, T={T_ns} ns, omega0*T = {cfg.omega0 * T_test:.2f} rad)")

# Convergence with n_steps
ns_vals = [5, 10, 20, 40, 80, 200]
errs = []
for ns in ns_vals:
    U_pwc_ns = build_propagator_pwc(H0_mat, Hc_mat, np.zeros(ns), T_test, ns)
    errs.append(np.max(np.abs(U_pwc_ns - U_sesolve_mat)))
    print(f"  n_steps = {ns:4d}  ->  max|error| = {errs[-1]:.3e}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Left: pulse shape visualisation
example_pulse = 0.2 * np.random.default_rng(0).standard_normal(n_steps)
tlist_plot = np.linspace(0, T_ns, n_steps)
axes[0].step(tlist_plot, example_pulse, where='post', lw=2, color='steelblue')
axes[0].set_xlabel('Time (ns)')
axes[0].set_ylabel('Pulse amplitude $u_k$')
axes[0].set_title(rf'Piecewise-constant pulse, $n_{{\rm steps}}={n_steps}$, $\delta t = {T_ns/n_steps:.1f}$ ns')

# Right: PWC convergence
axes[1].loglog(ns_vals, errs, 'o-', lw=2, color='darkorange')
# Show O(1/n_steps) reference line
ns_arr = np.array(ns_vals, dtype=float)
ref = errs[0] * ns_vals[0] / ns_arr
axes[1].loglog(ns_arr, ref, 'k--', label=r'$\propto 1/n_{\rm steps}$ (Lie–Trotter)')
axes[1].set_xlabel(r'$n_{\rm steps}$')
axes[1].set_ylabel('Max propagator error')
axes[1].set_title('PWC propagator convergence vs time slices')
axes[1].legend()

plt.tight_layout()
plt.show()

**Observation.** The PWC error scales as $\mathcal{O}(\delta t) = \mathcal{O}(1/n_{\rm steps})$,
consistent with the first-order Lie–Trotter product formula.  For the GRAPE JAX backend,
the Cayley approximant $(\mathbf{I} + iH\delta t/2)^{-1}(\mathbf{I} - iH\delta t/2)$
is used instead: it is also $\mathcal{O}(\delta t^2)$ accurate and
guaranteed unitary by construction, which stabilizes gradient computation.

---
*__Lab note.__ In experiment, a piecewise-constant pulse is uploaded to an
arbitrary waveform generator (AWG) as a list of amplitudes at a fixed sample
rate (typically 1–2 GS/s).  The AWG applies a reconstruction filter (sinc or
Gaussian), which band-limits the pulse and introduces a small deviation from
the ideal PWC shape.  This filtering can be pre-compensated by including the
filter response in the optimization model.*

<a id="exercises"></a>
## Exercises

**Exercise 1.** Compute the gate fidelity between `snap_operator(N, thetas)` and
`snap_operator(N, thetas + 0.1)` where `thetas = np.zeros(N); thetas[2] = np.pi`.  
Then repeat for errors of 0.01, 0.05, 0.1, 0.5, 1.0 rad and plot $1 - F$ vs
angle error.  What is the functional form?

**Exercise 2.** Use `np.argmax` to find the drive amplitude $\varepsilon$ that
maximizes $F$ in the scan from Section 3.  Then double `n_scan` and rerun the
scan — does the best $\varepsilon$ change?

In [ ]:
# Exercise 1 — fidelity vs angle error
# YOUR CODE HERE

In [ ]:
# Exercise 2 — best epsilon from scan
# YOUR CODE HERE

In [ ]:
# Solution — Exercise 1
thetas_base = np.zeros(N)
thetas_base[2] = np.pi
U_base = snap_operator(N, thetas_base)

errors_ex1 = np.array([0.001, 0.005, 0.01, 0.05, 0.1, 0.2, 0.5, 1.0])
infids_ex1 = []
for de in errors_ex1:
    th = thetas_base.copy()
    th[2] += de
    infids_ex1.append(1.0 - gate_fidelity(snap_operator(N, th), U_base))

# Analytic: 1-F = sin^2(de/2) * 4/N^2  [for single-level error, one level contributes e^{i de} - 1]
analytic = 4 * np.sin(errors_ex1 / 2)**2 / N**2

fig, ax = plt.subplots(figsize=(6, 4))
ax.loglog(errors_ex1, infids_ex1, 'o-', label='Numerical')
ax.loglog(errors_ex1, analytic, '--', label=r'Analytic: $4\sin^2(\delta\theta/2)/N^2$')
ax.set_xlabel(r'Angle error $\delta\theta$ (rad)')
ax.set_ylabel(r'Infidelity $1-F$')
ax.set_title('SNAP infidelity vs single-level phase error')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Solution — Exercise 2
best_idx_ex2 = int(np.argmax(fidelities_scan))
print(f"Best index: {best_idx_ex2}")
print(f"Best epsilon/(2pi): {epsilon_GHz_vals[best_idx_ex2]:.4f} GHz")
print(f"Best fidelity: F = {fidelities_scan[best_idx_ex2]:.6f}")